# Langchain Agents Integration

* So far our application has following two options,
    * A simple  RAG option - we retrieve the embeddings from the vector db using user entered text and fixed number of documents and augment the LLM reponse with this context. 
    * Optimized query otpion (chains) - we use LLM to create the query using user text and then use the LLM generated query to retrieve the embeddings from the database. 
* These options produce great results for simple queries, but for advanced metadata specific queries to fails to retrieve accurate response. 
* For that we'll need agentic option that will allow an LLM to reason and use different tools to retrieve and respond to user queries. 

## Import Libraries

In [10]:
from pathlib import Path
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain.schema import Document
from langchain_openai import OpenAIEmbeddings,ChatOpenAI
from langchain.agents import load_tools, initialize_agent
from langchain.agents import AgentType
from langgraph.prebuilt import create_react_agent
from langgraph.errors import GraphRecursionError


import os
import pandas as pd
import numpy as np
import chromadb

## Initialize DB

In [11]:
embedding_function = OpenAIEmbeddings()
vector_db = Chroma(persist_directory="../db/dunder_bot", embedding_function=embedding_function)

In [12]:
## lets verify the db connection
vector_db._collection.count()

29228

## Creating Tools    

## Tools

In [ ]:
from langchain.agents import tool

@tool
def get_total_seasons_of_the_office(text:str) -> int:
    """
    Returns the total number of seasons in the TV show The Office. Use only when the user’s question is clearly about the show. \
    The input should always be an empty string \
    and this function will always return a integer with total number of seasons. 
    This function cannot be used for any other season info
    Args:
        text (str): empty string

    Returns:
        int: integer representing total number of season in "The Office"
    """
    return 9

@tool
def get_season_info_of_the_office(text:str) -> int:
    pass
    

## Initializing Agent

In [14]:
## list of tools for the agent to use
custom_tools = [get_total_seasons_of_the_office]

## initialize llm model
llm_model = "gpt-3.5-turbo"
llm = ChatOpenAI(temperature=0, model=llm_model)

## initialize agent
agent = initialize_agent(
    llm=llm,
    agent=AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    tools=custom_tools,
    verbose=True,
    handle_parsing_errors=True
)

## Running Agent

In [15]:

try:
    response = agent.invoke("how many seasons are there in the office?"),
        # {"recursion_limit": recursion_limit},
except GraphRecursionError:
    print("Agent stopped due to max iterations.")



> Entering new AgentExecutor chain...
Thought: The user is asking about the total number of seasons in the TV show "The Office". I should use the tool `get_total_seasons_of_the_office` to find the answer. 

Action:
```
{
  "action": "get_total_seasons_of_the_office",
  "action_input": ""
}
``` 


Observation: 9
Thought:Final Answer: 9

> Finished chain.


In [7]:
response

({'input': 'how many seasons are there in the office?',
  'output': 'The Office has 9 seasons.'},)

In [8]:

try:
    response = agent.invoke("how many seasons are there in the country India?"),
        # {"recursion_limit": recursion_limit},
except GraphRecursionError:
    print("Agent stopped due to max iterations.")



> Entering new AgentExecutor chain...
Thought: This question is not about the TV show "The Office," so I cannot use the tool to get the total number of seasons of that show. I will need to provide a different type of response based on general knowledge.

Final Answer: I cannot determine the number of seasons in the country India as it does not relate to the TV show "The Office."

> Finished chain.


In [9]:
response

({'input': 'how many seasons are there in the country India?',
  'output': 'I cannot determine the number of seasons in the country India as it does not relate to the TV show "The Office."'},)